In [1]:
%load_ext autoreload 
%autoreload 2

In [4]:
import torch
from torch import nn


# CLIP vs SigLIP

# Example data

x = torch.Tensor([
    [1, 2, 3], [
        4 ,5, 6
    ]
])

y = torch.Tensor([
    [1, 2, 3], [
    5, 5, 7]
])
x = x / x.norm(dim=-1, keepdim=True)
y = y / y.norm(dim=-1, keepdim=True)

def clip(x, y, temperature=1.0):
    # x: [batch_size, embed_dim]
    # y: [batch_size, embed_dim]

    logit_scale = torch.exp(torch.Tensor([temperature]))

    # Normalize the vectors
    x = x / x.norm(dim=-1, keepdim=True)
    y = y / y.norm(dim=-1, keepdim=True)

    # Compute the cosine similarity
    logits = logit_scale * (x @ y.T)

    # Labels
    labels = torch.arange(x.shape[0], device=x.device)

    # Compute the loss
    loss_1 = nn.functional.cross_entropy(logits, labels)
    loss_2 = nn.functional.cross_entropy(logits.T, labels)
    loss = (loss_1 + loss_2) / 2
    return loss

def siglip(x, y, temperature=1.0, bias=1):
    # x: [batch_size, embed_dim]
    # y: [batch_size, embed_dim]

    logit_scale = torch.exp(torch.log(torch.tensor(1 / temperature))) 
    logit_bias = torch.Tensor([bias])
    targets = torch.eye(x.shape[0], device=x.device) * 2 - 1

    # Normalize the vectors
    x = x / x.norm(dim=-1, keepdim=True)
    y = y / y.norm(dim=-1, keepdim=True)

    # Compute the cosine similarity
    logits = logit_scale * (x @ y.T) + logit_bias

    # Compute the loss
    loss = nn.functional.binary_cross_entropy_with_logits(logits, targets)
    return loss


clip(x, y), siglip(x, y)


(tensor(0.6566), tensor(2.0996))

In [ ]:
from brain_image.model.loss import SigLipLoss



SigLipLoss(init_temperature=1., init_bias=1., max_scale=10000)(x, y)


/home/gasparyanartur/dev/brain-image-implementation/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/gasparyanartur/dev/brain-image-implementation/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadat

(tensor(2.0996, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>),
 tensor([[1.0000, 0.9670],
         [0.9746, 0.9965]]))